In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:

import re
from typing import NamedTuple, Literal, Optional

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
import seaborn as sns
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import torch
from tqdm.auto import tqdm, trange

from src.data import get_electrode_df, add_metadata_features

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
epochs_path = "outputs/epochs_preprocessed/EC260_epo.fif"
speech_responsive_path = "outputs/causal4/find_speech_responsive/EC260_results.csv"
As_path = "outputs/causal4/unify_As/results.csv"
A_decoders_path = "outputs/causal4/unify_As/unified_decoders.pt"
outdir = "."

window_size = 0.3  # seconds

include_rois = ["superiortemporal", "postcentral", "precentral", "supramarginal", "inferiortemporal", "middletemporal",
                "parsopercularis", "caudalmiddlefrontol", "lateralorbitofrontal", "temporalpole", "inferiorparietal",
                "rostralmiddlefrontal", "parstriangularis"]

In [ ]:
subject_name = re.findall("(EC[\d]+)_epo", epochs_path)[0]

In [ ]:
epochs = mne.read_epochs(epochs_path)
epochs.metadata = add_metadata_features(epochs.metadata)

In [ ]:
electrode_df = pd.read_csv(speech_responsive_path)
electrode_df

In [ ]:
As_df = pd.read_csv(As_path).query("subject == @subject_name")
As_df["tmin"] = As_df["smin"] / epochs.info["sfreq"] + epochs.tmin
As_df["tmax"] = As_df["smax"] / epochs.info["sfreq"] + epochs.tmin
As_df

In [ ]:
A_data = torch.load(A_decoders_path)

## Prepare A control values

In [ ]:
A_controls = {}

A_control_pipeline = make_pipeline(
    StandardScaler(),
    PCA(n_components=6)
)

for population in tqdm(As_df.itertuples(), total=len(As_df), leave=False, unit="A-population"):
    population_key = (population.subject, population.population_name, population.phoneme_pair)

    A_electrode_idxs = A_data["populations"][population_key]
    decoder_outcomes = A_data["held_out_outcomes"][population_key]

    # Estimate PCA on the non-held-out (training) trials
    pca_train_idxs = epochs.metadata[~epochs.metadata.index.isin(decoder_outcomes.epoch_idx)].index

    pca_train_data = epochs[pca_train_idxs].get_data(picks=A_electrode_idxs)[:, :, population.smin:population.smax]
    pca = clone(A_control_pipeline)
    pca.fit(pca_train_data.reshape(len(pca_train_data), -1))

    pca_test_idxs = decoder_outcomes.epoch_idx.unique()
    pca_test_data = epochs[pca_test_idxs].get_data(picks=A_electrode_idxs)[:, :, population.smin:population.smax]
    pca_test_data = pca.transform(pca_test_data.reshape(len(pca_test_data), -1))
    pca_test_df = pd.DataFrame(pca_test_data, columns=[f"pca_{i}" for i in range(pca_test_data.shape[1])],
                               index=pca_test_idxs).rename_axis("epoch_idx")
    
    A_controls[population_key] = pca_test_df.reset_index()

In [ ]:
control_plot_df = []
for population_key, control_data in A_controls.items():
    decoder_outcomes = A_data["held_out_outcomes"][population_key]
    
    merged_data = pd.merge(decoder_outcomes.groupby("epoch_idx").decoder_proba.mean().reset_index(),
                           control_data, on="epoch_idx", suffixes=("", "_control"))
    control_plot_df.append(merged_data.assign(population=str(population_key)))

In [ ]:
sns.lmplot(data=pd.concat(control_plot_df), x="decoder_proba", y="pca_0",
           col="population", col_wrap=3, height=4)

## Run search

In [ ]:
def run_B_searchlight(A_population: NamedTuple,
                      A_electrode_idxs: list[int],
                      A_outcomes: pd.DataFrame,
                      A_control_outcomes: pd.DataFrame,
                      measure: Literal["pearsonr", "spearmanr"] = "spearmanr",

                      window_size: float = 0.3,
                      window_tmin: float = 0.3,
                      window_tmax: float = 2.5,
                      AB_padding: Optional[float] = 0.1,):
    """
    Args:
        window_size: float
            Size of the B site window in seconds.
        window_tmax: float
            Maximum time in seconds for the right edge of the B window.
        AB_padding: float
            Minimum duration in seconds between the right edge of the A window
            and the left edge of the B window. This will override the `window_tmin`
            default.
    """
    # First compute average decoder outcomes within site, across folds
    A_outcomes = A_outcomes.groupby("epoch_idx").decoder_proba.mean()

    # reindex control data to match
    A_control_outcomes = A_control_outcomes.set_index("epoch_idx").pca_0.reindex(A_outcomes.index)

    epochs_ = epochs[A_outcomes.index]
    md = epochs_.metadata
    assert md is not None
    #### Everything beneath this line should now work on `epochs_`, which
    #### is the subset of trials for which we have decoder outcomes.

    # Select epochs which have lexical evidence toward the left phoneme.
    mask_left = md.lexical_evidence == 0

    epochs_left = epochs_[mask_left]
    epochs_right = epochs_[~mask_left]

    # balanced left/right
    assert len(epochs_left) == len(epochs_right)
    # balanced stimulus steps
    assert epochs_left.metadata.resampled.value_counts().nunique() == 1
    assert epochs_right.metadata.resampled.value_counts().nunique() == 1

    epochs_left_data = epochs_left.get_data()
    epochs_right_data = epochs_right.get_data()

    this_window_tmin = window_tmin
    if AB_padding is not None:
        this_window_tmin = max(A_population.tmax + AB_padding,
                               this_window_tmin)
    # TODO ensure phase?
    window_size_samp = int(window_size * epochs.info["sfreq"])
    window_start_samp = epochs.time_as_index(this_window_tmin)[0]
    window_end_samp = epochs.time_as_index(window_tmax)[0] + 1  # inclusive

    window_starts = np.arange(
        window_start_samp,
        window_end_samp - window_size_samp + 1,
        window_size_samp
    )
    window_ends = window_starts + window_size_samp

    ret = {}

    for window_start, window_end in zip(window_starts, window_ends):
        window_data_left = epochs_left_data[:, :, window_start:window_end]
        window_data_right = epochs_right_data[:, :, window_start:window_end]
        # average across time
        window_data_left = window_data_left.mean(axis=-1)
        window_data_right = window_data_right.mean(axis=-1)

        results, pca_control_results, stim_control_results = {}, {}, {}
        for side, window_data_mean, A_outcome_side, pca_control_side, stim_control_side in [
            ("left", window_data_left, A_outcomes[mask_left], A_control_outcomes[mask_left], md.resampled[mask_left]),
            ("right", window_data_right, A_outcomes[~mask_left], A_control_outcomes[~mask_left], md.resampled[~mask_left]),
        ]:
            if side == "left":
                p_gt_phoneme = 1 - A_outcome_side
            else:
                p_gt_phoneme = A_outcome_side

            if measure == "spearmanr":
                side_test = [spearmanr(window_data_mean[:, i], p_gt_phoneme)
                                for i in range(window_data_mean.shape[1])]
                corr, pval = zip(*side_test)

                pca_control_side_test = [spearmanr(window_data_mean[:, i], pca_control_side)
                                             for i in range(window_data_mean.shape[1])]
                pca_control_corr, pca_control_pval = zip(*pca_control_side_test)

                stim_control_side_test = [spearmanr(window_data_mean[:, i], stim_control_side)
                                           for i in range(window_data_mean.shape[1])]
                stim_control_corr, stim_control_pval = zip(*stim_control_side_test)
            elif measure == "pearsonr":
                raise NotImplementedError("Pearson's r is not implemented yet.")

            results[side] = (corr, pval)
            pca_control_results[side] = (pca_control_corr, pca_control_pval)
            stim_control_results[side] = (stim_control_corr, stim_control_pval)

        ret[window_start] = pd.DataFrame({
            "subject": A_population.subject,
            "population_name": A_population.population_name,
            "phoneme_pair": A_population.phoneme_pair,

            "electrode_idx": np.arange(len(results["left"][0])),
            "electrode_in_A": np.isin(np.arange(len(results["left"][0])),
                                    A_electrode_idxs),

            "corr_left": results["left"][0],
            "p_val_left": results["left"][1],
            "corr_right": results["right"][0],
            "p_val_right": results["right"][1],

            "pca_control_corr_left": pca_control_results["left"][0],
            "pca_control_p_val_left": pca_control_results["left"][1],
            "pca_control_corr_right": pca_control_results["right"][0],
            "pca_control_p_val_right": pca_control_results["right"][1],

            "stim_control_corr_left": stim_control_results["left"][0],
            "stim_control_p_val_left": stim_control_results["left"][1],
            "stim_control_corr_right": stim_control_results["right"][0],
            "stim_control_p_val_right": stim_control_results["right"][1],

            "window_start_samp": window_start,
            "window_end_samp": window_end,
            "window_start": epochs.times[window_start],
            "window_end": epochs.times[window_end],
        })

    return pd.concat(ret.values(), ignore_index=True)

In [ ]:
B_results = []

for population in tqdm(As_df.itertuples(), total=len(As_df), leave=False, unit="A-population"):
    population_key = (population.subject, population.population_name, population.phoneme_pair)

    A_electrode_idxs = A_data["populations"][population_key]
    decoder_outcomes = A_data["held_out_outcomes"][population_key]
    control_outcomes = A_controls[population_key]
    B_results.append(run_B_searchlight(population, A_electrode_idxs, 
                                       decoder_outcomes, control_outcomes))

In [ ]:
B_results_df = pd.concat(B_results, ignore_index=True)
# Only retain results for which we have electrode metadata
B_results_df = electrode_df.merge(B_results_df, on=["subject", "electrode_idx"], how="inner")

In [ ]:
# Only retain results for speech-responsive electrodes
B_results_df = B_results_df[B_results_df["speech_responsive"]]
# Only retain results for included ROIs
B_results_df = B_results_df[B_results_df["roi"].isin(include_rois)]

left_is_best = B_results_df[["p_val_left", "p_val_right"]].idxmin(axis=1) == "p_val_left"
B_results_df["left_is_best"] = left_is_best
B_results_df["p_val_min"] = B_results_df[["p_val_left", "p_val_right"]].min(axis=1)
B_results_df.loc[left_is_best, "pca_control_p_val_min"] = B_results_df.loc[left_is_best, "pca_control_p_val_left"]
B_results_df.loc[left_is_best, "stim_control_p_val_min"] = B_results_df.loc[left_is_best, "stim_control_p_val_left"]
B_results_df.loc[~left_is_best, "stim_control_p_val_min"] = B_results_df.loc[~left_is_best, "stim_control_p_val_right"]
B_results_df.loc[~left_is_best, "pca_control_p_val_min"] = B_results_df.loc[~left_is_best, "pca_control_p_val_right"]

B_results_df["p_val_min_log"] = -np.log10(B_results_df["p_val_min"])
B_results_df["stim_control_p_val_min_log"] = -np.log10(B_results_df["stim_control_p_val_min"])
B_results_df["pca_control_p_val_min_log"] = -np.log10(B_results_df["pca_control_p_val_min"])

In [ ]:
ax = sns.scatterplot(data=B_results_df, x="p_val_min_log", y="stim_control_p_val_min_log")
ax.plot(np.linspace(0, 10, 100), np.linspace(0, 10, 100), color="black", linestyle="--", alpha=0.5)

In [ ]:
ax = sns.scatterplot(data=B_results_df, x="p_val_min_log", y="pca_control_p_val_min_log")
ax.plot(np.linspace(0, 10, 100), np.linspace(0, 10, 100), color="black", linestyle="--", alpha=0.5)

## Save

In [ ]:
B_results_df.to_csv(f"{outdir}/{subject_name}_results.csv", index=False)